### np.cross

In [1]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for elem in a:
        result.extend(flatten(elem))
    return result


def _product(shape):
    p = 1
    for s in shape:
        p *= s
    return p


def _build_nested(shape, getter, prefix=()):
    if len(shape) == 0:
        return getter(prefix)
    return [_build_nested(shape[1:], getter, prefix + (i,)) for i in range(shape[0])]


def _get_item(a, idx):
    cur = a
    for i in idx:
        cur = cur[i]
    return cur


def _normalize_axis(axis, ndim):
    if axis < 0:
        axis += ndim
    if axis < 0 or axis >= ndim:
        raise ValueError(f"axis {axis} is out of bounds for array of dimension {ndim}")
    return axis


def _cross_2vec(u, v):
    if len(u) == 2 and len(v) == 2:
        return u[0] * v[1] - u[1] * v[0]
    if len(u) == 2 and len(v) == 3:
        u = [u[0], u[1], 0]
    elif len(u) == 3 and len(v) == 2:
        v = [v[0], v[1], 0]
    elif len(u) != 3 or len(v) != 3:
        raise ValueError("cross product is defined only for vectors of dimension 2 or 3")
    return [
        u[1] * v[2] - u[2] * v[1],
        u[2] * v[0] - u[0] * v[2],
        u[0] * v[1] - u[1] * v[0],
    ]


def _cross_flat_vectors(a, b):
    return _cross_2vec(a, b)


def _cross_last_axis(a, b):
    shape_a = get_shape(a)
    shape_b = get_shape(b)
    ndim_a = len(shape_a)
    ndim_b = len(shape_b)

    if ndim_a == 0 or ndim_b == 0:
        raise ValueError("scalar inputs are not allowed")

    if shape_a[:-1] != shape_b[:-1]:
        raise ValueError("shapes must match except for the last axis")

    if shape_a[-1] not in (2, 3) or shape_b[-1] not in (2, 3):
        raise ValueError("last axis must have length 2 or 3")

    if ndim_a == 1 and ndim_b == 1:
        return _cross_flat_vectors(a, b)

    out_shape = shape_a[:-1]

    def getter(prefix):
        vec_a = _get_item(a, prefix)
        vec_b = _get_item(b, prefix)
        return _cross_flat_vectors(vec_a, vec_b)

    return _build_nested(out_shape, getter)


def np_cross(a, b, axisa=-1, axisb=-1, axisc=-1, axis=None):
    
    shape_a = get_shape(a)
    shape_b = get_shape(b)
    ndim_a = len(shape_a)
    ndim_b = len(shape_b)

    if axis is not None:
        axisa = axis
        axisb = axis
        axisc = axis

    axisa = _normalize_axis(axisa, ndim_a)
    axisb = _normalize_axis(axisb, ndim_b)

    if shape_a[axisa] not in (2, 3) or shape_b[axisb] not in (2, 3):
        raise ValueError("cross product vectors must have dimension 2 or 3")

    if ndim_a == 1 and ndim_b == 1 and axisa == 0 and axisb == 0:
        return _cross_flat_vectors(a, b)

    if axisa != ndim_a - 1 or axisb != ndim_b - 1:
        raise NotImplementedError("Only last-axis cross product is implemented in this version")

    return _cross_last_axis(a, b)

### test cases 

In [2]:
print(np_cross([1, 2, 3], [4, 5, 6]))
print(np_cross([1, 2], [4, 5]))
print(np_cross([1, 2], [4, 5, 6]))
print(np_cross([1, 2, 3], [4, 5]))
print(np_cross([[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [1, 2, 3]]))
print(np_cross([[[1, 2, 3]], [[4, 5, 6]]], [[[7, 8, 9]], [[1, 2, 3]]]))
print(np_cross([[1, 2], [3, 4]], [[5, 6], [7, 8]]))

[-3, 6, -3]
-3
[12, -6, -3]
[-15, 12, -3]
[[-6, 12, -6], [3, -6, 3]]
[[[-6, 12, -6]], [[3, -6, 3]]]
[-4, -4]


In [3]:
print(np_cross([1], [2]))

ValueError: cross product vectors must have dimension 2 or 3

In [4]:
print(np_cross([1, 2, 3, 4], [5, 6, 7, 8]))

ValueError: cross product vectors must have dimension 2 or 3

In [5]:
print(np_cross([[1, 2], [3]], [[4, 5], [6, 7]]))

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [6]:
print(np_cross(3, [1, 2, 3]))

ValueError: axis -1 is out of bounds for array of dimension 0

In [7]:
print(np_cross([[1, 2, 3]], [[4, 5, 6]], axis=0))

ValueError: cross product vectors must have dimension 2 or 3